<a href="https://colab.research.google.com/github/krithika168/cse/blob/main/AI_quiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q gradio tensorflow pillow numpy

In [ ]:

import gradio as gr
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2,
    preprocess_input,
    decode_predictions
)

# Load pretrained CNN model
model = MobileNetV2(weights="imagenet")

questions = [
    ("What does AI stand for?",
     ["Artificial Intelligence", "Automatic Internet", "Advanced Input"],
     "Artificial Intelligence"),

    ("Which language is used for ML?",
     ["Java", "Python", "HTML"], "Python"),

    ("What does CNN stand for?",
     ["Computer Network", "Central Network",
      "Convolutional Neural Network"], "Convolutional Neural Network"),

    ("Which library is used for deep learning?",
     ["Pandas", "TensorFlow", "Matplotlib"], "TensorFlow"),

    ("Which algorithm is used for classification?",
     ["Decision Tree", "Linear Search", "SVM"], "SVM"),

    ("Which library is used for Machine Learning?",
     ["Scikit-learn", "Bootstrap", "React"], "Scikit-learn"),

    ("What is supervised learning?",
     ["Learning with labelled data", "Learning without data",
      "Random learning"], "Learning with labelled data"),

    ("Which algorithm is used for regression?",
     ["Binary Search","Bubble Sort","Linear Regression"],
     "Linear Regression"),

    ("What is a dataset?",
     ["Collection of data", "A website", "A computer"],
     "Collection of data"),

    ("What is classification?",
     ["Sorting files","Predicting categories","Deleting data"],
     "Predicting categories"),

    ("Which method divides data into training and testing sets?",
     ["Train-test split", "HTML parsing", "Data deletion"],
     "Train-test split"),

    ("What is overfitting?",
     ["Model has no data","Model is deleted","Model memorises training data"], "Model memorises training data"),

    ("Which layer extracts features in CNN?",
     ["Convolutional Layer", "Output Layer", "Input Layer"],
     "Convolutional Layer"),

    ("What is the full form of ReLU?",
     ["Random Learning Unit","Rectified Linear Unit",
      "Regular Linear Unit"], "Rectified Linear Unit"),

    ("What is the purpose of pooling?",
     ["Reduce feature map size", "Increase image size",
      "Delete images"], "Reduce feature map size"),

    ("Which model is used in Fruit Freshness Detection?",
     ["Linear Regression", "Decision Tree","MobileNetV2"],
     "MobileNetV2"),

    ("What are the two outputs of Fruit Freshness Detection?",
     ["Fruit type and freshness", "Price and weight",
      "Size and shape"], "Fruit type and freshness"),

    ("Which tool creates the web interface?",
     ["MySQL", "Excel","Gradio"], "Gradio"),

    ("Which function trains a CNN model?",
     ["model.fit()", "model.delete()", "model.close()"],
     "model.fit()"),

    ("Which function predicts an image class?",
     ["model.predict()", "model.remove()", "model.rename()"],
     "model.predict()")
]


def predict_image(image):
    if image is None:
        return "Please upload an image."

    image = image.resize((224, 224))
    image = np.array(image)

    image = preprocess_input(image)
    image = np.expand_dims(image, axis=0)

    predictions = model.predict(image, verbose=0)
    results = decode_predictions(predictions, top=3)[0]

    output = "CNN MODEL PREDICTION\n\n"

    for i, (_, label, confidence) in enumerate(results, 1):
        output += (
            f"{i}. {label.replace('_', ' ').title()} : "
            f"{confidence * 100:.2f}%\n"
        )

    return output


def show_question(index):
    if index >= len(questions):
        return (
            "Quiz completed!",
            gr.update(choices=[], value=None),
            gr.update(visible=False),
            gr.update(visible=True)
        )

    q = questions[index]

    return (
        q[0],
        gr.update(choices=q[1], value=None),
        gr.update(visible=True),
        gr.update(visible=False)
    )


def start_game():
    random.shuffle(questions)

    return (
        0, 0,
        "Quiz started!",
        *show_question(0)
    )


def check_answer(choice, score, index):

    if index >= len(questions):
        return score, index, "Quiz already completed!"

    if choice is None:
        return score, index, "Please select an answer!"

    correct_answer = questions[index][2]

    if choice == correct_answer:
        score += 1
        message = "Correct answer!"
    else:
        message = "Wrong answer! Correct answer: " + correct_answer

    return score, index + 1, message


def next_question(index):
    return show_question(index)


def finish_quiz(score):
    total = len(questions)
    percentage = (score / total) * 100

    return (
        f"Your final score: {score}/{total}\n"
        f"Percentage: {percentage:.2f}%\n"
        "Thank you for playing!"
    )


with gr.Blocks(theme=gr.themes.Soft()) as app:

    gr.Markdown("# 🧠 AI & CNN QUIZ CHALLENGE")
    gr.Markdown(
        "Test your AI, Machine Learning and CNN knowledge!"
    )

    with gr.Tab("CNN Image Prediction"):

        gr.Markdown("### Upload an image for CNN classification")

        image_input = gr.Image(
            type="pil",
            label="Upload Image"
        )

        predict_btn = gr.Button(
            "Predict Image",
            variant="primary"
        )

        prediction_output = gr.Textbox(
            label="CNN Prediction",
            lines=5
        )

        predict_btn.click(
            predict_image,
            inputs=image_input,
            outputs=prediction_output
        )

    with gr.Tab("AI Quiz Game"):

        score = gr.State(0)
        index = gr.State(0)

        question = gr.Textbox(
            label="Question",
            interactive=False
        )

        options = gr.Radio(
            label="Choose your answer",
            choices=[]
        )

        result = gr.Textbox(
            label="Result",
            interactive=False
        )

        score_display = gr.Number(
            label="Score",
            value=0,
            interactive=False
        )

        final_result = gr.Textbox(
            label="Final Result",
            interactive=False,
            visible=False
        )

        with gr.Row():
            start = gr.Button("Start Quiz", variant="primary")
            submit = gr.Button("Submit Answer")
            next_btn = gr.Button("Next Question")
            finish = gr.Button("Finish Quiz")

        start.click(
            start_game,
            outputs=[
                score, index, result,
                question, options, submit, final_result
            ]
        ).then(
            lambda s: s,
            inputs=score,
            outputs=score_display
        )

        submit.click(
            check_answer,
            inputs=[options, score, index],
            outputs=[score, index, result]
        ).then(
            lambda s: s,
            inputs=score,
            outputs=score_display
        )

        next_btn.click(
            next_question,
            inputs=index,
            outputs=[
                question, options, submit, final_result
            ]
        )

        finish.click(
            finish_quiz,
            inputs=score,
            outputs=final_result
        ).then(
            lambda: gr.update(visible=True),
            outputs=final_result
        )


app.launch(share=True)

/tmp/ipykernel_2301/302868677.py:178: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6b09f5c2523bccf4d2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
